# 🎯 EDA — XGBoost: Análise de Risco OLA
## Predictfy × Locaweb — FIAP Challenge 2026

Notebook exploratório: treina o modelo e visualiza curvas, SHAP e distribuições de risco.

**Input:** `data/processed/incidents_features.parquet`
**Módulo:** `src.models.xgboost_model`


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from sklearn.metrics import (
    precision_recall_curve, roc_curve,
    roc_auc_score, average_precision_score,
    confusion_matrix
)
from src.models.xgboost_model import load_data, train, FEATURES, TARGET

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Carregando dados e treinando modelo (aguarde ~30s)...")
df = load_data()
results = train(df)

threshold  = results["threshold_otm"]
y_test     = results["y_test"]
y_prob     = results["y_prob_final"]
shap_values = results["shap_values"]
shap_abs   = results["shap_abs"]
print("Pronto!")

## 1. Curvas Precision-Recall e ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p, r, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
axes[0].plot(r, p, color="#1a73e8", lw=2, label=f"Modelo (PR-AUC={pr_auc:.3f})")
axes[0].axhline(y=y_test.mean(), color="gray", ls="--", label=f"Baseline ({y_test.mean():.4f})")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Curva Precision-Recall", fontsize=12, fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3)

fp_, tp_, _ = roc_curve(y_test, y_prob)
roc = roc_auc_score(y_test, y_prob)
axes[1].plot(fp_, tp_, color="#1a73e8", lw=2, label=f"Modelo (ROC={roc:.3f})")
axes[1].plot([0,1],[0,1],"k--", label="Aleatório", lw=1)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("Curva ROC", fontsize=12, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 2. Matriz de Confusão (Threshold Otimizado)

In [ ]:
y_pred = (y_prob >= threshold).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NAO (0)", "SIM (1)"],
            yticklabels=["NAO (0)", "SIM (1)"], ax=ax)
ax.set_xlabel("Predito"); ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão — Threshold {threshold:.4f}", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"TP (violações capturadas): {tp}")
print(f"FN (violações perdidas):   {fn}")
print(f"FP (falsos alarmes):       {fp}")
print(f"TN (OK confirmados):       {tn}")

## 3. SHAP — Feature Importance

In [ ]:
import shap

fig_size = max(6, len(FEATURES) // 2)
fig, ax = plt.subplots(figsize=(9, fig_size))
shap_df = sorted(zip(FEATURES, shap_abs), key=lambda x: x[1])
feats, vals = zip(*shap_df)
ax.barh(feats, vals, color="#1a73e8", edgecolor="white")
ax.set_xlabel("Importância média |SHAP|")
ax.set_title("Feature Importance — XGBoost OLA Risk", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Distribuição de Probabilidades por Classe

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(y_prob[y_test == 0], bins=50, alpha=0.7, color="#5ac8fa", label="Real: NAO", density=True)
ax.hist(y_prob[y_test == 1], bins=20, alpha=0.85, color="#ff2d55", label="Real: SIM", density=True)
ax.axvline(threshold, color="black", ls="--", label=f"Threshold = {threshold:.3f}")
ax.set_xlabel("Probabilidade")
ax.set_title("Distribuição das Probabilidades por Classe Real", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Métricas Completas

In [ ]:
m = results["metricas"]
print("Métricas no conjunto de teste:")
print(f"  Recall (violações capturadas): {m['recall_violacao']:.4f}")
print(f"  Precision:                     {m['precision_violacao']:.4f}")
print(f"  F1-Score:                      {m['f1_violacao']:.4f}")
print(f"  ROC-AUC test:                  {m['roc_auc']:.4f}")
print(f"  PR-AUC test:                   {m['pr_auc']:.4f}")
print(f"  ROC-AUC CV (5-fold):           {m['roc_auc_cv_mean']:.4f} ± {m['roc_auc_cv_std']:.4f}")
print(f"  PR-AUC CV (5-fold):            {m['pr_auc_cv_mean']:.4f} ± {m['pr_auc_cv_std']:.4f}")
print()
print(f"Top 5 features (SHAP):")
for feat in results["feat_imp_list"][:5]:
    print(f"  {feat['rank']:>2}. {feat['feature']:<25}  {feat['shap_mean_abs']:.5f}")